# Reward Engineering for Game-Playing Agents

**GOLDS Project -- Notebook 04**

This notebook covers one of the most impactful (and underappreciated) parts of
deep reinforcement learning: **how you shape the reward signal that your agent
learns from**. A small change in reward design can be the difference between an
agent that masters a game and one that sits in a corner doing nothing.

We will work through five topics:

1. The reward shaping problem (sparse vs. dense, reward hacking, potential-based shaping)
2. Reward clipping -- the classic Atari approach
3. Running reward normalization with `VecNormalize`
4. Writing custom reward wrappers
5. How GOLDS exposes all of this via the `reward_regime` config option

**Prerequisites:** familiarity with the Gymnasium `Wrapper` API, basic probability, and PPO at a high level.

In [ ]:
# Shared imports used throughout the notebook.
import numpy as np

try:
    import matplotlib
    matplotlib.use("Agg")  # non-interactive backend for headless environments
    import matplotlib.pyplot as plt
    %matplotlib inline
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not found -- plots will be skipped.")

try:
    import gymnasium as gym
    HAS_GYM = True
except ImportError:
    HAS_GYM = False
    print("gymnasium not found -- wrapper demos will use stubs.")

np.random.seed(42)

---
## 1. The Reward Shaping Problem

### Sparse vs. Dense Rewards

Most classic games give the agent a **sparse** reward signal: you get points
only when you score, collect a coin, or defeat an enemy.  Between those events
the reward is zero.  For a learning algorithm that relies on reward to figure
out what is good, long stretches of zero signal make credit assignment
extremely difficult.

A **dense** reward, by contrast, provides feedback at every timestep.  For
example, giving a small positive reward proportional to how far right Mario
moved since the last frame turns a sparse platformer reward into a dense one.

### Reward Hacking

Dense rewards are powerful, but they come with a trap: the agent will
optimize *exactly* what you measure, and that might not be what you intended.
Classic examples:

| Game | Intended behaviour | Reward hack |
|---|---|---|
| CoastRunners (boat race) | Finish the race | Drive in circles collecting boost pads -- never finishes |
| Sonic the Hedgehog | Complete the level | Find a spot where the x-position oscillates, farming progress reward |
| Tetris | Clear lines | Delay line clears to farm piece-placement reward |

The lesson: **adding auxiliary reward terms changes the optimal policy**, and
the new optimum may be undesirable.

### Potential-Based Reward Shaping

Ng, Harada & Russell (1999) proved a beautiful result: if you define the
shaping reward as a *difference of potentials*, the set of optimal policies
is **unchanged**.

Formally, let $\Phi : \mathcal{S} \to \mathbb{R}$ be a bounded real-valued
function of state (the *potential*).  Define the shaping reward:

$$
F(s, s') = \gamma\, \Phi(s') - \Phi(s)
$$

where $\gamma$ is the discount factor.  Then any policy that is optimal under
the original reward $R$ is also optimal under $R + F$, and vice versa.

**Intuition:** the $\gamma\,\Phi(s') - \Phi(s)$ term telescopes along any
trajectory.  Over an infinite horizon the total shaping reward is
$-\Phi(s_0)$, a constant that does not depend on the policy, so the ranking
of policies is preserved.

This gives us a principled recipe for dense reward:
1. Pick a potential function that encodes progress (e.g., $\Phi(s) = x\text{-position}$).
2. Compute the shaping reward $F(s,s')$ at each step.
3. Add $F$ to the environment reward.

Because the theorem guarantees policy-invariance, you get faster learning
without introducing reward hacking -- *as long as* your $\Phi$ is truly
state-only and bounded.

---
## 2. Reward Clipping

The DeepMind Atari paper (Mnih et al., 2015) introduced a brutally simple
trick: **clip every reward to {-1, 0, +1}** by taking the sign.

```python
clipped_reward = np.sign(raw_reward)  # -1, 0, or +1
```

### Why clip?

| Benefit | Explanation |
|---|---|
| **Stability** | Reward magnitudes vary wildly across Atari games (Pong: +/-1, Space Invaders: 5--200, Breakout: 1--7). Clipping puts all games on the same scale, so a single set of hyperparameters works everywhere. |
| **Bounded gradients** | With rewards in [-1, 1] the value-function targets stay small, preventing gradient explosions early in training. |

### Why *not* clip?

| Drawback | Explanation |
|---|---|
| **Lost magnitude** | Destroying a mothership worth 200 points looks the same as destroying a common alien worth 5 points.  The agent cannot learn to prioritise high-value targets. |
| **Distorted incentives** | In Breakout, clearing the last few bricks is worth the same as the first few, even though completing a screen grants a large bonus. |

In Stable-Baselines3 the `ClipRewardEnv` wrapper (part of `AtariWrapper`)
handles this.  In GOLDS the `clip_reward: true` config option enables it.

In [ ]:
# Demonstrate the effect of reward clipping on a synthetic reward stream.
#
# We simulate 2000 timesteps of a game that produces:
#   - mostly zeros (no event)
#   - occasional small positive rewards (common enemy)
#   - rare large positive rewards (boss / bonus)
#   - occasional negative rewards (taking damage)

rng = np.random.default_rng(42)
n_steps = 2000

# Generate raw rewards
raw_rewards = np.zeros(n_steps)
for i in range(n_steps):
    roll = rng.random()
    if roll < 0.02:       # 2% chance: big bonus
        raw_rewards[i] = rng.choice([50, 100, 200])
    elif roll < 0.10:     # 8% chance: small positive
        raw_rewards[i] = rng.choice([5, 10, 20])
    elif roll < 0.13:     # 3% chance: damage
        raw_rewards[i] = rng.choice([-10, -25])
    # else: 0 (no event)

# Clip to {-1, 0, +1}
clipped_rewards = np.sign(raw_rewards)

print(f"Raw rewards   -- min: {raw_rewards.min():.0f}, max: {raw_rewards.max():.0f}, "
      f"mean: {raw_rewards.mean():.2f}, nonzero: {np.count_nonzero(raw_rewards)}")
print(f"Clipped rewards -- unique values: {np.unique(clipped_rewards)}, "
      f"mean: {clipped_rewards.mean():.2f}, nonzero: {np.count_nonzero(clipped_rewards)}")

In [ ]:
# Plot raw vs. clipped reward distributions side by side.

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # --- Raw rewards histogram ---
    ax = axes[0]
    nonzero_raw = raw_rewards[raw_rewards != 0]
    ax.hist(nonzero_raw, bins=30, color="steelblue", edgecolor="black", alpha=0.8)
    ax.set_title("Raw Rewards (nonzero only)")
    ax.set_xlabel("Reward")
    ax.set_ylabel("Count")
    ax.axvline(0, color="gray", linestyle="--", linewidth=0.8)

    # --- Clipped rewards bar chart ---
    ax = axes[1]
    values, counts = np.unique(clipped_rewards, return_counts=True)
    colors = ["#d9534f" if v < 0 else ("#5cb85c" if v > 0 else "#cccccc") for v in values]
    ax.bar([str(int(v)) for v in values], counts, color=colors, edgecolor="black")
    ax.set_title("Clipped Rewards")
    ax.set_xlabel("Reward")
    ax.set_ylabel("Count")

    fig.suptitle("Effect of Reward Clipping", fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()
else:
    print("Skipping plot (matplotlib not available).")

Notice how the wide dynamic range (5 to 200) collapses into a single
`+1` bin after clipping.  The agent can no longer distinguish a 200-point
bonus from a 5-point pickup -- but every game now has the same reward scale.

---
## 3. VecNormalize -- Running Reward Statistics

An alternative to hard clipping is **online normalization**.  Stable-Baselines3
provides `VecNormalize`, a vectorized-environment wrapper that maintains a
running mean and standard deviation of the discounted return and divides each
reward by the running $\sigma$.

Concretely, `VecNormalize` keeps an exponential moving estimator of
$\text{Var}[R_{\text{discounted}}]$ and normalizes:

$$
r_{\text{norm}} = \text{clip}\!\left(\frac{r}{\sqrt{\text{Var} + \epsilon}},\; -c,\; c\right)
$$

where $c$ is a configurable clip bound (default 10) and $\epsilon$ avoids
division by zero.

### When it helps

- **Continuous-control tasks** where reward magnitudes are unknown ahead of time.
- **Games with gradual reward inflation** (e.g., score multipliers in later levels).
- When you want to preserve *relative* reward magnitudes (unlike clipping).

### When it hurts

- **Non-stationary rewards:** if the reward distribution shifts drastically
  mid-training (e.g., reaching a new game level where scores jump 10x),
  the running statistics lag behind and normalization temporarily distorts
  the signal.
- **Evaluation:** you must freeze the running stats at evaluation time
  (`training=False`), otherwise evaluation episodes update the mean and
  contaminate training statistics.
- **Checkpointing:** you must save and reload the `VecNormalize` state
  alongside the model, or the normalization statistics reset on resume.

In [ ]:
# Demonstrate running reward normalization on synthetic sequences.

class RunningMeanStd:
    """Minimal Welford running mean/variance (mirrors SB3 implementation)."""

    def __init__(self, epsilon: float = 1e-4):
        self.mean = 0.0
        self.var = 1.0
        self.count = epsilon

    def update(self, x: np.ndarray) -> None:
        batch_mean = np.mean(x)
        batch_var = np.var(x)
        batch_count = len(x)

        delta = batch_mean - self.mean
        total_count = self.count + batch_count

        self.mean = self.mean + delta * batch_count / total_count
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        m2 = m_a + m_b + delta ** 2 * self.count * batch_count / total_count
        self.var = m2 / total_count
        self.count = total_count


def normalize_rewards(raw: np.ndarray, gamma: float = 0.99, clip: float = 10.0) -> np.ndarray:
    """Simulate VecNormalize's online reward normalization."""
    rms = RunningMeanStd()
    discounted_return = 0.0
    normalized = np.zeros_like(raw, dtype=np.float64)

    for i, r in enumerate(raw):
        discounted_return = r + gamma * discounted_return
        # Update stats with the discounted return (as SB3 does)
        rms.update(np.array([discounted_return]))
        # Normalize the raw reward by the running std
        std = np.sqrt(rms.var + 1e-8)
        normalized[i] = np.clip(r / std, -clip, clip)

    return normalized


# --- Scenario A: Stationary rewards ---
stationary_raw = raw_rewards.copy()  # reuse from Section 2
stationary_norm = normalize_rewards(stationary_raw)

# --- Scenario B: Non-stationary (reward jump at step 1000) ---
nonstationary_raw = raw_rewards.copy()
nonstationary_raw[1000:] *= 10  # simulate reaching a high-score level
nonstationary_norm = normalize_rewards(nonstationary_raw)

print("Stationary -- raw range: [{:.0f}, {:.0f}], normalized range: [{:.2f}, {:.2f}]".format(
    stationary_raw.min(), stationary_raw.max(),
    stationary_norm.min(), stationary_norm.max()))
print("Non-stationary -- raw range: [{:.0f}, {:.0f}], normalized range: [{:.2f}, {:.2f}]".format(
    nonstationary_raw.min(), nonstationary_raw.max(),
    nonstationary_norm.min(), nonstationary_norm.max()))

In [ ]:
# Plot normalized vs. raw for both scenarios.

if HAS_MPL:
    fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex="col")

    # Row 0: stationary
    axes[0, 0].plot(stationary_raw, linewidth=0.4, color="steelblue")
    axes[0, 0].set_title("Stationary -- Raw")
    axes[0, 0].set_ylabel("Reward")

    axes[0, 1].plot(stationary_norm, linewidth=0.4, color="darkorange")
    axes[0, 1].set_title("Stationary -- Normalized")
    axes[0, 1].set_ylabel("Norm. Reward")

    # Row 1: non-stationary
    axes[1, 0].plot(nonstationary_raw, linewidth=0.4, color="steelblue")
    axes[1, 0].set_title("Non-stationary -- Raw")
    axes[1, 0].set_xlabel("Timestep")
    axes[1, 0].set_ylabel("Reward")
    axes[1, 0].axvline(1000, color="red", linestyle="--", linewidth=0.8, label="regime shift")
    axes[1, 0].legend(fontsize=8)

    axes[1, 1].plot(nonstationary_norm, linewidth=0.4, color="darkorange")
    axes[1, 1].set_title("Non-stationary -- Normalized")
    axes[1, 1].set_xlabel("Timestep")
    axes[1, 1].set_ylabel("Norm. Reward")
    axes[1, 1].axvline(1000, color="red", linestyle="--", linewidth=0.8, label="regime shift")
    axes[1, 1].legend(fontsize=8)

    fig.suptitle("VecNormalize: Stationary vs. Non-stationary Rewards", fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()
else:
    print("Skipping plot (matplotlib not available).")

In the **stationary** case, the normalized rewards quickly settle into a
stable range.  In the **non-stationary** case, notice the spike right after
step 1000: the running variance has not yet caught up to the 10x jump, so
the normalized rewards temporarily overshoot.  This lag is the main practical
downside of `VecNormalize` for game environments.

---
## 4. Custom Reward Wrappers

When clipping is too lossy and normalization is too fragile, you can write
a custom Gymnasium `Wrapper` that reshapes the reward however you like.

A common pattern for platformers: **x-position progress reward**.  At each
timestep the agent receives `new_x - old_x` as a bonus, encouraging
rightward movement.  This is an instance of potential-based shaping with
$\Phi(s) = x(s)$ (the x-coordinate of the player).

Below we implement a minimal version.

In [ ]:
if HAS_GYM:
    class XProgressRewardWrapper(gym.Wrapper):
        """Add x-position progress as a dense reward bonus.

        The wrapper reads the player's x-position from the info dict
        (key: 'x_pos') and adds (new_x - old_x) * scale to the reward.

        This is potential-based shaping with Phi(s) = x_pos(s) * scale,
        so (assuming gamma ~ 1) the set of optimal policies is preserved.
        """

        def __init__(self, env: gym.Env, x_pos_key: str = "x_pos", scale: float = 0.01):
            super().__init__(env)
            self.x_pos_key = x_pos_key
            self.scale = scale
            self._last_x = 0.0

        def reset(self, **kwargs):
            obs, info = self.env.reset(**kwargs)
            self._last_x = info.get(self.x_pos_key, 0.0)
            return obs, info

        def step(self, action):
            obs, reward, terminated, truncated, info = self.env.step(action)

            current_x = info.get(self.x_pos_key, self._last_x)
            progress = (current_x - self._last_x) * self.scale
            self._last_x = current_x

            shaped_reward = reward + progress
            return obs, shaped_reward, terminated, truncated, info


    # --- Quick demo with a dummy environment ---
    class FakePlatformerEnv(gym.Env):
        """Tiny 1-D platformer for demonstration purposes."""

        metadata = {"render_modes": ["rgb_array"]}

        def __init__(self):
            super().__init__()
            self.observation_space = gym.spaces.Box(low=0, high=255, shape=(4,), dtype=np.uint8)
            self.action_space = gym.spaces.Discrete(3)  # left, stay, right
            self.x_pos = 0.0
            self.step_count = 0

        def reset(self, seed=None, options=None):
            super().reset(seed=seed)
            self.x_pos = 0.0
            self.step_count = 0
            return np.zeros(4, dtype=np.uint8), {"x_pos": self.x_pos}

        def step(self, action):
            # action: 0=left, 1=stay, 2=right
            self.x_pos += (action - 1) * 5.0  # move by -5, 0, or +5
            self.step_count += 1

            # Sparse reward: +100 only when reaching x=50
            reward = 100.0 if self.x_pos >= 50.0 else 0.0
            terminated = self.x_pos >= 50.0
            truncated = self.step_count >= 100

            obs = np.zeros(4, dtype=np.uint8)
            return obs, reward, terminated, truncated, {"x_pos": self.x_pos}


    # Run a short episode with and without the wrapper.
    env_raw = FakePlatformerEnv()
    env_shaped = XProgressRewardWrapper(FakePlatformerEnv(), scale=0.1)

    actions = [2, 2, 2, 1, 2, 0, 2, 2, 2, 2]  # mostly move right

    print("Step | Action | Raw Reward | Shaped Reward")
    print("-----|--------|------------|-------------")

    obs_r, info_r = env_raw.reset()
    obs_s, info_s = env_shaped.reset()

    for i, a in enumerate(actions):
        _, r_raw, _, _, _ = env_raw.step(a)
        _, r_shaped, _, _, _ = env_shaped.step(a)
        print(f"  {i:2d} |   {a}    |   {r_raw:6.1f}   |   {r_shaped:6.2f}")

    env_raw.close()
    env_shaped.close()
else:
    print("gymnasium not installed -- skipping wrapper demo.")
    print()
    print("The XProgressRewardWrapper reads info['x_pos'] at each step")
    print("and adds (new_x - old_x) * scale to the reward.")
    print("This converts a sparse goal reward into a dense progress signal.")

With the raw environment, the agent sees nothing but zeros until it finally
reaches x=50.  With the shaped wrapper, every rightward step produces a small
positive reward, and every leftward step produces a small negative reward.
The agent now has a learning signal at every timestep.

**Key design rules for custom reward wrappers:**

1. **Use potential-based shaping** to avoid changing the optimal policy.
2. **Keep the scale small** relative to the environment reward so the agent
   does not ignore the true objective.
3. **Log both raw and shaped rewards** so you can diagnose reward hacking
   during training.

### 4.1 Extended Reward Shaping — Beyond X-Position

GOLDS v3 extends `PlatformerRewardWrapper` with three additional reward signals beyond x-position progress:

| Signal | Config field | Description |
|--------|-------------|-------------|
| **Death penalty** | `death_penalty` | Negative reward on `terminated=True` (not truncation). Teaches the agent that dying is bad. |
| **Collectible bonus** | `collectible_reward_scale` | Reward for collecting rings (Sonic) or coins (Mario). Uses delta: `scale * (new_count - old_count)`. |
| **Time penalty** | `time_penalty` | Small per-step negative reward. Encourages the agent to complete levels faster instead of standing still. |

All four signals (x-progress + death + collectible + time) are summed and added to the raw environment reward. Each is tracked separately in the `info` dict for debugging:

```python
info["raw_reward"]          # Original game reward
info["shaped_x_progress"]   # scale * (x_new - x_old)
info["shaped_death"]        # death_penalty if terminated else 0
info["shaped_collectible"]  # collectible_scale * delta_rings
info["shaped_time"]         # time_penalty (constant per step)
```

**Important:** `clip_reward: false` must be set in the config when using reward shaping. Otherwise, `RetroPreprocessing` clips all shaped rewards to {-1, 0, +1}, destroying the carefully designed signals.

In [ ]:
# Demonstrate all reward signals working together
if HAS_GYM:
    # Simulate a Sonic episode with all shaping signals
    n_steps = 50
    x_positions = np.concatenate([
        np.linspace(0, 200, 30),    # moving right
        np.full(10, 200),           # stuck at obstacle
        np.linspace(200, 100, 10),  # pushed back / died
    ])
    rings = np.concatenate([
        np.cumsum(np.random.choice([0, 0, 0, 1, 1], size=30)),  # collecting rings
        np.cumsum(np.random.choice([0, 0, 0, 0, 1], size=10)),
        np.zeros(10),  # lost rings on death
    ]) + 0

    # Shaping parameters
    x_scale = 0.1
    death_pen = -1.0
    collect_scale = 0.1
    time_pen = -0.001

    raw_rewards = np.ones(n_steps)  # constant raw reward of 1.0
    x_shaped = np.diff(x_positions, prepend=x_positions[0]) * x_scale
    collect_shaped = np.maximum(0, np.diff(rings, prepend=rings[0])) * collect_scale
    time_shaped = np.full(n_steps, time_pen)
    death_shaped = np.zeros(n_steps)
    death_shaped[-1] = death_pen  # die on last step

    total = raw_rewards + x_shaped + collect_shaped + time_shaped + death_shaped

    if HAS_MPL:
        fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

        axes[0].plot(x_positions[:n_steps], label="X position", color="steelblue")
        axes[0].set_ylabel("X position")
        axes[0].legend()

        axes[1].bar(range(n_steps), x_shaped, alpha=0.7, label="X progress", color="green")
        axes[1].bar(range(n_steps), collect_shaped, alpha=0.7, bottom=x_shaped, label="Collectibles", color="gold")
        axes[1].bar(range(n_steps), time_shaped, alpha=0.5, label="Time penalty", color="gray")
        axes[1].bar(range(n_steps), death_shaped, alpha=0.8, label="Death penalty", color="red")
        axes[1].set_ylabel("Shaping components")
        axes[1].legend(fontsize=8)

        axes[2].plot(total, label="Total reward (raw + all shaping)", color="purple", linewidth=1.5)
        axes[2].axhline(0, color="gray", linestyle="--", linewidth=0.5)
        axes[2].set_ylabel("Total reward")
        axes[2].set_xlabel("Timestep")
        axes[2].legend()

        fig.suptitle("Extended Reward Shaping: All Signals Combined", fontsize=14, y=1.01)
        fig.tight_layout()
        plt.show()
else:
    print("gymnasium not available -- skipping demo")

---
## 4.2 Intrinsic Curiosity — Random Network Distillation (RND)

Sometimes even dense reward shaping is not enough. When the agent reaches a hard obstacle (a wide gap in Sonic, a tricky jump in Mario), the reward signal goes flat — no forward progress means no x-position reward, and the agent has no incentive to try new strategies.

**RND (Random Network Distillation)** solves this by adding an exploration bonus. The idea is elegant:

1. Take a **fixed random neural network** (the "target") — it maps observations to random embeddings
2. Train a **predictor network** to match the target's outputs
3. **Intrinsic reward** = prediction error (MSE between predictor and target)

States the agent has seen many times are easy to predict (low error, low bonus). **Novel states produce high prediction error** (high bonus), encouraging the agent to explore.

```
Target Network (frozen random weights)
    obs → [CNN] → embedding_target
                                        → MSE = intrinsic reward
Predictor Network (trained)
    obs → [CNN] → embedding_pred
```

In GOLDS, RND is implemented as a `VecEnvWrapper` (`RNDRewardWrapper`) that adds intrinsic reward inline during `step()`. This is important — it must happen before PPO computes advantages, so a callback-based approach would be too late.

**When to use RND:**
- Platformers with hard obstacles (Sonic, Mario, Mega Man)
- Sparse-reward exploration games (Montezuma's Revenge)

**When NOT to use RND:**
- Fighting games — health-delta reward is already dense
- Puzzle games — score-based reward is sufficient
- Sonic caveat: dynamic backgrounds (timer, moving clouds) can produce spurious intrinsic reward. Keep `rnd_reward_scale` small (0.01).

```yaml
training:
  rnd_enabled: true
  rnd_reward_scale: 0.01    # Keep small to avoid overwhelming extrinsic signal
  rnd_learning_rate: 1e-4
```

In [ ]:
# Demonstrate how RND intrinsic reward decays with familiarity
import torch
from golds.training.rnd import RNDNetwork, RunningMeanStd

target = RNDNetwork()
predictor = RNDNetwork()
optimizer = torch.optim.Adam(predictor.parameters(), lr=1e-3)

# Freeze target
for p in target.parameters():
    p.requires_grad = False

# Simulate: show same observation repeatedly, watch intrinsic reward decay
obs = torch.randn(1, 1, 84, 84)  # one fixed observation
rewards = []

for step in range(200):
    with torch.no_grad():
        target_feat = target(obs)
    pred_feat = predictor(obs)
    mse = ((target_feat - pred_feat) ** 2).mean()
    rewards.append(mse.item())

    # Train predictor
    optimizer.zero_grad()
    mse.backward()
    optimizer.step()

if HAS_MPL:
    plt.figure(figsize=(10, 4))
    plt.plot(rewards, color="darkorange", linewidth=1.5)
    plt.xlabel("Training steps on the same observation")
    plt.ylabel("Intrinsic reward (MSE)")
    plt.title("RND: Intrinsic reward decays as the predictor learns")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print(f"Initial intrinsic reward: {rewards[0]:.4f}")
print(f"After 200 steps:          {rewards[-1]:.4f}")
print(f"Reduction:                {(1 - rewards[-1]/rewards[0])*100:.1f}%")

---
## 5. GOLDS Reward Regimes

The GOLDS project centralises reward engineering decisions in a single config
field: `environment.reward_regime`.  It accepts three values:

| `reward_regime` | What happens | When to use |
|---|---|---|
| `"clipped"` | Rewards are clipped to {-1, 0, +1} via `np.sign()` inside `RetroPreprocessing` or SB3's `ClipRewardEnv`. | Default for Atari and retro games. Best for stability when you do not need magnitude information. |
| `"raw"` | No reward transformation.  The agent sees the original game score deltas. | Use when reward magnitudes carry important information (e.g., distinguishing a 200-point boss kill from a 5-point minion). |
| `"normalized"` | `VecNormalize(norm_obs=False, norm_reward=True, clip_reward=10.0)` is applied. Running statistics are saved/loaded with the model. | Use for games with unknown or shifting reward scales, or when you want to preserve relative magnitudes without manual tuning. |

The mapping lives in `golds/environments/factory.py` inside `EnvironmentFactory.create()`.
Here is the relevant logic (simplified):

```python
# In EnvironmentFactory.create():
if reward_regime == "normalized":
    from stable_baselines3.common.vec_env import VecNormalize
    vec_env = VecNormalize(vec_env, norm_obs=False, norm_reward=True, clip_reward=10.0)
```

For `"clipped"`, the clipping happens inside the per-environment preprocessing
wrapper (`RetroPreprocessing.clip_reward` or `AtariWrapper`), which is
controlled by `clip_reward: true` in the config.

For `"raw"`, both clipping and normalization are skipped.

In [ ]:
# Load a real GOLDS config and inspect the reward_regime mapping.

import sys
from pathlib import Path

# Make GOLDS importable if running from the notebooks/ directory.
project_root = Path(".").resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

try:
    from golds.config.loader import ConfigLoader

    loader = ConfigLoader(config_dir=project_root / "configs")

    # Load configs for several games and display their reward regimes.
    game_ids = ["space_invaders", "breakout", "pong",
                "super_mario_bros", "sonic_the_hedgehog", "tetris"]

    header = (f"{'Game':<25} {'Platform':<8} {'reward_regime':<12} {'clip_reward':<12} "
              f"{'death_penalty':<15} {'collect_scale':<15} {'time_penalty':<13} {'rnd_enabled'}")
    print(header)
    print("-" * len(header))

    for gid in game_ids:
        try:
            cfg = loader.load_game(gid)
            env = cfg.environment
            trn = cfg.training
            death_pen = getattr(env, "death_penalty", "N/A")
            collect_sc = getattr(env, "collectible_reward_scale", "N/A")
            time_pen = getattr(env, "time_penalty", "N/A")
            rnd = getattr(trn, "rnd_enabled", False)
            print(f"{gid:<25} {env.platform:<8} "
                  f"{env.reward_regime:<12} {str(env.clip_reward):<12} "
                  f"{str(death_pen):<15} {str(collect_sc):<15} {str(time_pen):<13} {rnd}")
        except FileNotFoundError:
            print(f"{gid:<25} (config not found)")

except ImportError as e:
    print(f"Could not import GOLDS config loader: {e}")
    print()
    print("Expected reward config values from GOLDS configs:")
    print()
    print(f"{'Game':<25} {'Platform':<8} {'regime':<10} {'clip':<6} "
          f"{'death_pen':<12} {'collect_sc':<12} {'time_pen':<10} {'rnd'}")
    print("-" * 95)
    print("  space_invaders         atari    clipped    True   N/A          N/A          N/A        False")
    print("  breakout               atari    clipped    True   N/A          N/A          N/A        False")
    print("  pong                   atari    clipped    True   N/A          N/A          N/A        False")
    print("  super_mario_bros       retro    clipped    True   -1.0         0.1          -0.001     False")
    print("  sonic_the_hedgehog     retro    raw        False  -1.0         0.1          -0.001     True")
    print("  tetris                 retro    clipped    True   N/A          N/A          N/A        False")
    print()
    print("NOTE: When using extended reward shaping (death_penalty, collectible_reward_scale,")
    print("time_penalty), set clip_reward: false to prevent reward signals from being destroyed.")
    print("Sonic uses reward_regime: raw + clip_reward: false because it relies on shaped signals.")

In [ ]:
# Show how each reward_regime maps to wrapper behavior.

def describe_reward_pipeline(regime: str) -> str:
    """Describe the reward processing pipeline for a given regime."""
    steps = ["Game emulator produces raw score delta"]

    if regime == "clipped":
        steps.append("ClipRewardEnv / RetroPreprocessing: r = np.sign(r)")
        steps.append("Result: rewards in {-1, 0, +1}")
    elif regime == "raw":
        steps.append("No reward transformation applied")
        steps.append("Result: agent sees original score deltas")
    elif regime == "normalized":
        steps.append("VecNormalize: track running mean/var of discounted returns")
        steps.append("VecNormalize: r_norm = clip(r / sqrt(var + eps), -10, 10)")
        steps.append("Result: rewards normalized to ~unit variance")
    else:
        steps.append(f"Unknown regime: {regime}")

    return "\n  -> ".join(steps)


for regime in ["clipped", "raw", "normalized"]:
    print(f"reward_regime = \"{regime}\"")
    print(f"  -> {describe_reward_pipeline(regime)}")
    print()

### Choosing a Regime for Your Experiment

A practical decision tree:

1. **Start with `clipped`.**  It is the most stable default and matches the
   published Atari baselines.
2. If the agent learns but plateaus because it cannot distinguish high-value
   events, switch to `normalized`.
3. If you have domain knowledge about the reward scale and want full control,
   use `raw` and tune the value-function loss coefficient (`vf_coef`) or
   learning rate to compensate for large reward magnitudes.
4. If you write a custom reward wrapper (like `XProgressRewardWrapper` above),
   you can pair it with any regime -- just make sure the shaping bonus and
   the regime do not conflict (e.g., do not clip a carefully scaled bonus
   down to +/-1).

---
## Summary

| Technique | Preserves magnitude? | Preserves optimal policy? | Stable across games? |
|---|---|---|---|
| Reward clipping | No | No (changes ranking of actions with different magnitudes) | Yes |
| VecNormalize | Relative, yes | Approximately (normalization is not potential-based) | Mostly |
| Potential-based shaping | Yes (additive) | Yes (provably) | Depends on choice of $\Phi$ |
| Raw (no transform) | Yes | Yes | No (hyperparameters must be tuned per game) |

The right choice depends on what matters most for your experiment: stability,
faithfulness to the original game score, or learning speed.  GOLDS lets you
switch between these with a single config line:

```yaml
environment:
  reward_regime: clipped  # or 'raw' or 'normalized'
```

### References

- Ng, Harada, Russell. *Policy Invariance Under Reward Transformations.* ICML 1999.
- Mnih et al. *Human-level Control Through Deep Reinforcement Learning.* Nature 2015.
- Stable-Baselines3 documentation: [VecNormalize](https://stable-baselines3.readthedocs.io/en/master/guide/vec_envs.html#vecnormalize)